# Parallel Hyperparameter Search - Make That M4 Pro Sweat 🔥

Running multiple quantum GNN training jobs simultaneously using ProcessPoolExecutor.

**Strategy:**
- Use concurrent.futures.ProcessPoolExecutor for parallel training
- Import training function from external module to avoid pickling issues
- Run 6 configurations simultaneously across 12 cores
- Real-time progress monitoring and result aggregation

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import json
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
import time
from itertools import product
from tqdm.notebook import tqdm

# Import the training worker function
from parallel_train_worker import train_single_config

print(f"✓ Imports successful")
print(f"✓ Worker function loaded from parallel_train_worker.py")

## 1. Configuration

In [ ]:
# Paths
DATA_DIR = "/Users/priyanshudey/Code/Qunatum copy/othercode/data"
SAVE_DIR = "./parallel_hyperparam_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# Reproducibility
SEED = 42069

# Training parameters (fixed across all configs)
EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
BATCH_SIZE = 512

# Hyperparameter search space
HIDDEN_DIM_VALUES = [64, 128, 256]
N_QUBITS_VALUES = [6, 8, 10]
N_QLAYERS_VALUES = [4, 6, 8]
LEARNING_RATE_VALUES = [0.001, 0.0005, 0.0001]

# Parallel execution settings
NUM_PARALLEL_JOBS = 6  # Run 6 configs at once

total_configs = len(HIDDEN_DIM_VALUES) * len(N_QUBITS_VALUES) * len(N_QLAYERS_VALUES) * len(LEARNING_RATE_VALUES)

print("="*80)
print("PARALLEL HYPERPARAMETER SEARCH CONFIGURATION")
print("="*80)
print(f"Data directory:       {DATA_DIR}")
print(f"Save directory:       {SAVE_DIR}")
print(f"Batch size:           {BATCH_SIZE}")
print(f"Max epochs:           {EPOCHS}")
print(f"Early stopping:       {EARLY_STOPPING_PATIENCE} epochs")
print(f"\nHyperparameter Grid:")
print(f"  Hidden Dim:         {HIDDEN_DIM_VALUES}")
print(f"  N Qubits:           {N_QUBITS_VALUES}")
print(f"  N QLayers:          {N_QLAYERS_VALUES}")
print(f"  Learning Rates:     {LEARNING_RATE_VALUES}")
print(f"\nParallel Execution:")
print(f"  Concurrent jobs:    {NUM_PARALLEL_JOBS}")
print(f"  Total configs:      {total_configs}")
print(f"  Estimated time:     ~{total_configs / NUM_PARALLEL_JOBS * 5:.0f} minutes (5 min/config)")
print("="*80)

## 2. Generate All Configurations

In [ ]:
# Generate all hyperparameter combinations
configs = list(product(HIDDEN_DIM_VALUES, N_QUBITS_VALUES, N_QLAYERS_VALUES, LEARNING_RATE_VALUES))

print(f"Generated {len(configs)} configurations:")
print("\nFirst 5 configs:")
for i, (hd, nq, nql, lr) in enumerate(configs[:5], 1):
    print(f"  {i}: HD={hd}, Q={nq}, QL={nql}, LR={lr}")
print("  ...")
print(f"\nLast config:")
hd, nq, nql, lr = configs[-1]
print(f"  {len(configs)}: HD={hd}, Q={nq}, QL={nql}, LR={lr}")

## 3. Run Parallel Training 🔥

This cell will train all configurations in parallel. Progress will be shown in real-time.

In [ ]:
print("="*80)
print("STARTING PARALLEL HYPERPARAMETER SEARCH")
print("="*80)
print(f"\n🔥 Time to make that M4 Pro sweat! 💦\n")
print(f"Running {NUM_PARALLEL_JOBS} jobs in parallel...\n")

results = []
overall_start = time.time()

# Use ProcessPoolExecutor for parallel training
with ProcessPoolExecutor(max_workers=NUM_PARALLEL_JOBS) as executor:
    # Submit all jobs
    future_to_config = {}
    for i, (hd, nq, nql, lr) in enumerate(configs, 1):
        future = executor.submit(
            train_single_config,
            config_num=i,
            hidden_dim=hd,
            n_qubits=nq,
            n_qlayers=nql,
            learning_rate=lr,
            data_dir=DATA_DIR,
            save_dir=SAVE_DIR,
            seed=SEED,
            epochs=EPOCHS,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            batch_size=BATCH_SIZE
        )
        future_to_config[future] = (i, hd, nq, nql, lr)
    
    # Process results as they complete
    with tqdm(total=len(configs), desc="Overall Progress") as pbar:
        for future in as_completed(future_to_config):
            config_info = future_to_config[future]
            try:
                result = future.result()
                results.append(result)
                
                if result['status'] == 'success':
                    pbar.set_postfix({
                        'Last': f"#{result['config_num']}",
                        'AUC': f"{result['test_auc']:.4f}",
                        'Time': f"{result['training_time_sec']/60:.1f}m"
                    })
                else:
                    pbar.set_postfix({'Last': f"#{result['config_num']} FAILED"})
                    
            except Exception as e:
                print(f"\n⚠️  Config {config_info[0]} raised exception: {e}")
                results.append({
                    'config_num': config_info[0],
                    'hidden_dim': config_info[1],
                    'n_qubits': config_info[2],
                    'n_qlayers': config_info[3],
                    'learning_rate': config_info[4],
                    'status': 'failed',
                    'error': str(e)
                })
            
            pbar.update(1)

overall_time = time.time() - overall_start

print("\n" + "="*80)
print("🎉 PARALLEL SEARCH COMPLETE!")
print("="*80)
print(f"Total wall-clock time:    {overall_time/60:.2f} minutes")
print(f"Average time per config:  {overall_time/len(configs):.1f} seconds")
print(f"Configurations completed: {len(results)}/{len(configs)}")

## 4. Results Analysis

In [ ]:
# Separate successful and failed results
results_df = pd.DataFrame([r for r in results if r.get('status') == 'success'])
failed_df = pd.DataFrame([r for r in results if r.get('status') == 'failed'])

print("="*80)
print("RESULTS SUMMARY")
print("="*80)
print(f"✓ Successful:  {len(results_df)} configs")
print(f"✗ Failed:      {len(failed_df)} configs")

if len(failed_df) > 0:
    print("\nFailed Configurations:")
    print(failed_df[['config_num', 'hidden_dim', 'n_qubits', 'n_qlayers', 'learning_rate', 'error']])

if len(results_df) > 0:
    # Sort by test AUC
    results_df = results_df.sort_values('test_auc', ascending=False)
    
    # Save results
    results_df.to_csv(os.path.join(SAVE_DIR, 'parallel_hyperparam_results.csv'), index=False)
    
    print("\n" + "="*100)
    print("TOP 10 CONFIGURATIONS BY TEST AUC")
    print("="*100)
    top_10 = results_df.head(10)[['config_num', 'hidden_dim', 'n_qubits', 'n_qlayers', 
                                    'learning_rate', 'test_auc', 'test_accuracy', 'test_f1',
                                    'training_time_sec']]
    top_10_display = top_10.copy()
    top_10_display['training_time_sec'] = top_10_display['training_time_sec'].apply(lambda x: f"{x/60:.1f}m")
    print(top_10_display.to_string(index=False))
    
    # Best configuration details
    best = results_df.iloc[0]
    print("\n" + "="*80)
    print("🏆 BEST CONFIGURATION")
    print("="*80)
    print(f"Config Number:      #{int(best['config_num'])}")
    print(f"Hidden Dim:         {int(best['hidden_dim'])}")
    print(f"N Qubits:           {int(best['n_qubits'])}")
    print(f"N QLayers:          {int(best['n_qlayers'])}")
    print(f"Learning Rate:      {best['learning_rate']}")
    print(f"Model Parameters:   {int(best['model_params']):,}")
    print(f"\nPerformance Metrics:")
    print(f"Val AUC:            {best['val_auc']:.4f}")
    print(f"Test AUC:           {best['test_auc']:.4f}")
    print(f"Test Accuracy:      {best['test_accuracy']:.4f}")
    print(f"Test Precision:     {best['test_precision']:.4f}")
    print(f"Test Recall:        {best['test_recall']:.4f}")
    print(f"Test F1:            {best['test_f1']:.4f}")
    print(f"\nTraining Time:      {best['training_time_sec']/60:.2f} minutes")
    print("="*80)
    
    # Calculate parallel efficiency
    total_compute_time = results_df['training_time_sec'].sum()
    parallel_speedup = total_compute_time / overall_time
    
    print(f"\n⚡ PARALLEL EFFICIENCY")
    print(f"Total compute time:     {total_compute_time/60:.2f} minutes")
    print(f"Actual wall time:       {overall_time/60:.2f} minutes")
    print(f"Parallel speedup:       {parallel_speedup:.2f}x")
    print(f"Efficiency:             {(parallel_speedup/NUM_PARALLEL_JOBS)*100:.1f}%")
else:
    print("\n⚠️  No successful configurations to analyze!")

## 5. Comprehensive Visualizations

In [ ]:
if len(results_df) > 0:
    # Create comprehensive visualizations
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    # 1. Test AUC vs Hidden Dim
    ax = axes[0, 0]
    for lr in LEARNING_RATE_VALUES:
        lr_data = results_df[results_df['learning_rate'] == lr]
        if len(lr_data) > 0:
            grouped = lr_data.groupby('hidden_dim')['test_auc'].agg(['mean', 'std'])
            ax.errorbar(grouped.index, grouped['mean'], yerr=grouped['std'], 
                        marker='o', label=f'LR={lr}', capsize=5, linewidth=2, markersize=8)
    ax.set_xlabel('Hidden Dimension', fontsize=12, fontweight='bold')
    ax.set_ylabel('Test AUC', fontsize=12, fontweight='bold')
    ax.set_title('Test AUC vs Hidden Dimension', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 2. Test AUC vs N_QUBITS
    ax = axes[0, 1]
    for lr in LEARNING_RATE_VALUES:
        lr_data = results_df[results_df['learning_rate'] == lr]
        if len(lr_data) > 0:
            grouped = lr_data.groupby('n_qubits')['test_auc'].agg(['mean', 'std'])
            ax.errorbar(grouped.index, grouped['mean'], yerr=grouped['std'], 
                        marker='o', label=f'LR={lr}', capsize=5, linewidth=2, markersize=8)
    ax.set_xlabel('Number of Qubits', fontsize=12, fontweight='bold')
    ax.set_ylabel('Test AUC', fontsize=12, fontweight='bold')
    ax.set_title('Test AUC vs Number of Qubits', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 3. Test AUC vs N_QLAYERS
    ax = axes[0, 2]
    for lr in LEARNING_RATE_VALUES:
        lr_data = results_df[results_df['learning_rate'] == lr]
        if len(lr_data) > 0:
            grouped = lr_data.groupby('n_qlayers')['test_auc'].agg(['mean', 'std'])
            ax.errorbar(grouped.index, grouped['mean'], yerr=grouped['std'], 
                        marker='o', label=f'LR={lr}', capsize=5, linewidth=2, markersize=8)
    ax.set_xlabel('Number of Quantum Layers', fontsize=12, fontweight='bold')
    ax.set_ylabel('Test AUC', fontsize=12, fontweight='bold')
    ax.set_title('Test AUC vs Quantum Layers', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 4. Test AUC vs Learning Rate
    ax = axes[1, 0]
    lr_grouped = results_df.groupby('learning_rate')['test_auc'].agg(['mean', 'std'])
    ax.errorbar(lr_grouped.index, lr_grouped['mean'], yerr=lr_grouped['std'], 
                marker='o', capsize=5, capthick=2, linewidth=2, markersize=10, color='darkblue')
    ax.set_xlabel('Learning Rate', fontsize=12, fontweight='bold')
    ax.set_ylabel('Test AUC', fontsize=12, fontweight='bold')
    ax.set_title('Test AUC vs Learning Rate', fontsize=14, fontweight='bold')
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3)
    
    # 5. Training Time Distribution
    ax = axes[1, 1]
    training_times = results_df['training_time_sec'] / 60
    ax.hist(training_times, bins=20, edgecolor='black', alpha=0.7, color='coral')
    ax.axvline(training_times.mean(), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {training_times.mean():.1f} min')
    ax.set_xlabel('Training Time (minutes)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax.set_title('Training Time Distribution', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 6. Test AUC Distribution
    ax = axes[1, 2]
    ax.hist(results_df['test_auc'], bins=20, edgecolor='black', alpha=0.7, color='lightgreen')
    ax.axvline(results_df['test_auc'].mean(), color='darkgreen', linestyle='--', linewidth=2, 
               label=f'Mean: {results_df["test_auc"].mean():.3f}')
    ax.axvline(results_df['test_auc'].max(), color='red', linestyle='--', linewidth=2, 
               label=f'Best: {results_df["test_auc"].max():.3f}')
    ax.set_xlabel('Test AUC', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax.set_title('Test AUC Distribution', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'parallel_hyperparam_analysis.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Plots saved to {SAVE_DIR}/parallel_hyperparam_analysis.png")
else:
    print("⚠️  No data to visualize")

## 6. Detailed Statistical Analysis

In [ ]:
if len(results_df) > 0:
    print("="*100)
    print("HYPERPARAMETER IMPACT ANALYSIS")
    print("="*100)
    
    print("\n1. BY HIDDEN DIMENSION:")
    hd_summary = results_df.groupby('hidden_dim').agg({
        'test_auc': ['mean', 'std', 'max', 'min', 'count'],
        'training_time_sec': 'mean'
    }).round(4)
    hd_summary.columns = ['AUC_mean', 'AUC_std', 'AUC_max', 'AUC_min', 'count', 'avg_time_sec']
    print(hd_summary)
    
    print("\n2. BY NUMBER OF QUBITS:")
    qubits_summary = results_df.groupby('n_qubits').agg({
        'test_auc': ['mean', 'std', 'max', 'min', 'count'],
        'training_time_sec': 'mean'
    }).round(4)
    qubits_summary.columns = ['AUC_mean', 'AUC_std', 'AUC_max', 'AUC_min', 'count', 'avg_time_sec']
    print(qubits_summary)
    
    print("\n3. BY NUMBER OF QUANTUM LAYERS:")
    qlayers_summary = results_df.groupby('n_qlayers').agg({
        'test_auc': ['mean', 'std', 'max', 'min', 'count'],
        'training_time_sec': 'mean'
    }).round(4)
    qlayers_summary.columns = ['AUC_mean', 'AUC_std', 'AUC_max', 'AUC_min', 'count', 'avg_time_sec']
    print(qlayers_summary)
    
    print("\n4. BY LEARNING RATE:")
    lr_summary = results_df.groupby('learning_rate').agg({
        'test_auc': ['mean', 'std', 'max', 'min', 'count'],
        'training_time_sec': 'mean'
    }).round(4)
    lr_summary.columns = ['AUC_mean', 'AUC_std', 'AUC_max', 'AUC_min', 'count', 'avg_time_sec']
    print(lr_summary)
    
    # Save all summary statistics
    summary = {
        'best_auc': float(results_df['test_auc'].max()),
        'worst_auc': float(results_df['test_auc'].min()),
        'mean_auc': float(results_df['test_auc'].mean()),
        'std_auc': float(results_df['test_auc'].std()),
        'total_configs': len(results_df),
        'failed_configs': len(failed_df),
        'total_compute_time_min': float(results_df['training_time_sec'].sum()/60),
        'actual_wall_time_min': float(overall_time/60),
        'parallel_speedup': float(results_df['training_time_sec'].sum()/overall_time),
        'best_config': {
            'config_num': int(results_df.iloc[0]['config_num']),
            'hidden_dim': int(results_df.iloc[0]['hidden_dim']),
            'n_qubits': int(results_df.iloc[0]['n_qubits']),
            'n_qlayers': int(results_df.iloc[0]['n_qlayers']),
            'learning_rate': float(results_df.iloc[0]['learning_rate']),
            'test_auc': float(results_df.iloc[0]['test_auc'])
        }
    }
    
    with open(os.path.join(SAVE_DIR, 'parallel_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n✓ Summary statistics saved to {SAVE_DIR}/parallel_summary.json")
    print(f"✓ Full results saved to {SAVE_DIR}/parallel_hyperparam_results.csv")
    print(f"\n🎊 Analysis complete! Check {SAVE_DIR}/ for all results.")